# 🛡️ CICIDS2017 PortScan Intrusion Detection System
### Binary Classification of Network Traffic: BENIGN vs PortScan

This notebook provides an optimized, memory-efficient implementation of the PortScan detection pipeline using **CICIDS2017** network flows.

**Pipeline Overview:**
1. **Data Ingestion & Sanitization** (Handling infinite values, constant columns, null values)
2. **Balanced Sampling** (50,000 Benign + 50,000 PortScan samples)
3. **Exploratory Data Analysis (EDA)** (Targeted ports, flow duration, packet distribution)
4. **Feature Standardization & Selection** (`StandardScaler` + `SelectKBest` ANOVA F-test, k=20)
5. **Model Training & Comparison** (Random Forest, SVM, MLP Neural Network)
6. **Comprehensive Evaluation** (Confusion Matrices, Precision/Recall/F1, ROC-AUC, 5-Fold Cross Validation)

In [ ]:
import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

## 1. Memory-Efficient Data Loading & Preprocessing
PortScan attacks in CICIDS2017 are situated in `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv`. Loading this file directly avoids RAM exhaustion and provides abundant Benign and PortScan instances.

In [ ]:
file_path = 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
print(f'Loading {file_path}...')
df_raw = pd.read_csv(file_path)
df_raw.columns = df_raw.columns.str.strip()

# 1. Filter BENIGN and PortScan
filtered_df = df_raw[df_raw['Label'].isin(['BENIGN', 'PortScan'])].copy()
del df_raw
gc.collect()

# 2. Drop constant/single-value columns
constant_cols = [c for c in filtered_df.columns if filtered_df[c].nunique() <= 1]
filtered_df.drop(columns=constant_cols, inplace=True)

# 3. Handle Infinite and Missing values
filtered_df.replace([np.inf, -np.inf], np.nan, inplace=True)
filtered_df.dropna(inplace=True)

# 4. Label encoding (BENIGN -> 0, PortScan -> 1)
filtered_df['Label'] = filtered_df['Label'].replace({'BENIGN': 0, 'PortScan': 1})

# 5. Balanced Sampling (50,000 from each class)
df_benign = filtered_df[filtered_df['Label'] == 0].sample(n=50000, random_state=42)
df_portscan = filtered_df[filtered_df['Label'] == 1].sample(n=50000, random_state=42)
df = pd.concat([df_benign, df_portscan], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

del filtered_df, df_benign, df_portscan
gc.collect()

print('Balanced dataset shape:', df.shape)
print('Class distribution:\n', df['Label'].value_counts())

## 2. Exploratory Data Analysis (EDA)
Inspecting key differences between BENIGN traffic and PortScan attacks.

In [ ]:
# Top 20 Targeted Destination Ports
portscan_df = df[df['Label'] == 1]
top_ports = portscan_df['Destination Port'].value_counts().head(20)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_ports.index.astype(str), y=top_ports.values, palette='mako')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 Destination Ports Targeted by PortScan', fontsize=13, fontweight='bold')
plt.xlabel('Destination Port')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Flow Duration Distribution (KDE)
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df[df['Label'] == 0]['Flow Duration'], label='BENIGN', fill=True, color='teal')
sns.kdeplot(data=df[df['Label'] == 1]['Flow Duration'], label='PortScan', fill=True, color='crimson')
plt.title('Flow Duration Distribution (Microseconds)', fontsize=13, fontweight='bold')
plt.xlabel('Flow Duration')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average Packet Size: BENIGN vs PortScan
plot_df = df[['Label', 'Average Packet Size']].copy()
plot_df['Traffic Type'] = plot_df['Label'].replace({0: 'BENIGN', 1: 'PortScan'})

plt.figure(figsize=(8, 5))
sns.boxplot(x='Traffic Type', y='Average Packet Size', data=plot_df, palette=['#3b82f6', '#ef4444'])
plt.title('Average Packet Size - BENIGN vs PortScan', fontsize=13, fontweight='bold')
plt.show()
del plot_df

In [ ]:
# Forward vs Backward Packets Scatter Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(
    x='Total Fwd Packets', y='Total Backward Packets',
    data=df[df['Label'] == 0], color='royalblue', alpha=0.4, ax=axes[0]
)
axes[0].set_title('BENIGN Traffic Packets Distribution', fontweight='bold')
axes[0].set_xlim(-5, 150)
axes[0].set_ylim(-5, 150)

sns.scatterplot(
    x='Total Fwd Packets', y='Total Backward Packets',
    data=df[df['Label'] == 1], color='crimson', alpha=0.4, ax=axes[1]
)
axes[1].set_title('PortScan Traffic Packets Distribution', fontweight='bold')
axes[1].set_xlim(-5, 150)
axes[1].set_ylim(-5, 150)
plt.tight_layout()
plt.show()

## 3. Feature Scaling & Feature Selection (SelectKBest, k=20)
Standardizing numeric features and selecting top 20 attributes using ANOVA F-value (`f_classif`).

In [ ]:
X = df.drop(columns=['Label'])
y = df['Label']

# Standardize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Select top 20 features
selector = SelectKBest(score_func=f_classif, k=20)
X_new = selector.fit_transform(X_scaled, y)
selected_features = X.columns[selector.get_support()]

print('Top 20 Selected Features:')
for i, f in enumerate(selected_features, 1):
    print(f'  {i:2d}. {f}')

X_selected = pd.DataFrame(X_new, columns=selected_features)

# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, stratify=y, test_size=0.2, random_state=42
)
print(f'\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}')

## 4. Random Forest Feature Importance Plot

In [ ]:
rf_feat = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_feat.fit(X_train, y_train)

plt.figure(figsize=(10, 7))
pd.Series(rf_feat.feature_importances_, index=selected_features).sort_values().plot(kind='barh', color='darkcyan')
plt.title('Top 20 Features Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()

## 5. Model Training & Evaluation (Random Forest, SVM, MLP)
Training and comparing the three classifiers.

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42)
}

results = {}
y_probs = {}

for name, model in models.items():
    print(f'Training {name}...')
    if name == 'Support Vector Machine' and len(X_train) > 10000:
        sub_idx = np.random.RandomState(42).choice(len(X_train), size=10000, replace=False)
        model.fit(X_train.iloc[sub_idx], y_train.iloc[sub_idx])
    else:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_probs[name] = model.predict_proba(X_test)[:, 1]
    else:
        y_probs[name] = model.decision_function(X_test)

    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'Confusion Matrix': confusion_matrix(y_test, y_pred)
    }
    print(f'  Accuracy : {results[name]["Accuracy"]*100:.2f}%')
    print(f'  Precision: {results[name]["Precision"]:.4f}')
    print(f'  Recall   : {results[name]["Recall"]:.4f}')
    print(f'  F1-Score : {results[name]["F1-score"]:.4f}\n')

## 6. Confusion Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, metrics) in zip(axes, results.items()):
    sns.heatmap(metrics['Confusion Matrix'], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted Label (0=BENIGN, 1=PortScan)')
    ax.set_ylabel('Actual Label')
plt.tight_layout()
plt.show()

## 7. Model Performance Comparison Chart & ROC Curves

In [ ]:
# Model Performance Bar Chart
metrics_list = []
for model_name, metric_dict in results.items():
    metrics_list.append({
        'Model': model_name,
        'Accuracy': metric_dict['Accuracy'],
        'Precision': metric_dict['Precision'],
        'Recall': metric_dict['Recall'],
        'F1-Score': metric_dict['F1-score']
    })
pd.DataFrame(metrics_list).set_index('Model').plot(kind='bar', figsize=(11, 5), colormap='viridis')
plt.title('Model Performance Comparison (BENIGN vs PortScan)', fontsize=13, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0.95, 1.005)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# ROC Curves
plt.figure(figsize=(9, 6))
for name, prob in y_probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (PortScan Detection)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 5-Fold Stratified Cross-Validation on Random Forest

In [ ]:
cv_scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), X_selected, y, cv=5, scoring='accuracy')
print(f'Random Forest 5-Fold CV Accuracy: {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std() * 100:.2f}%)')